# DocBenchTR - A100 hardware control run

Re-runs the published pilot with the **same** model (`qwen2.5:7b-instruct`), the same 32k context, the same 36 questions and the **same cached PageIndex trees** as the local RTX 3070 Laptop run. Only the GPU differs.

Runtime > Change runtime type > **A100 GPU**, then Runtime > Run all. Results are copied to Google Drive (`MyDrive/docbenchtr_colab/`) and downloaded as a zip.

In [ ]:
# Parameters: reproduce with this immutable source revision.
REPO_COMMIT = '6360a35e2bb117414059a493261eb77b301d9088'

In [ ]:
# 1. GPU check: this notebook is meant for an A100.
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
assert gpu.returncode == 0, 'No GPU attached. Runtime > Change runtime type > A100 GPU.'
GPU_INFO = gpu.stdout.strip()
if 'A100' not in GPU_INFO:
    print('WARNING: not an A100. The run still works, but record which GPU was used.')

In [ ]:
# 2. Google Drive (results are copied here). Stops if Drive is not authorized.
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/docbenchtr_colab')
assert Path('/content/drive/MyDrive').exists(), 'Drive did not mount. Re-run this cell and approve access.'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive folder:', DRIVE_ROOT)

In [ ]:
# 3. Check out the immutable source revision recorded in the parameter cell.
REPO_URL = 'https://github.com/umitaltinozzz/docbenchtr'
REPO = Path('/content/docbenchtr')
assert 'DRIVE_ROOT' in globals(), 'Run the Drive cell first.'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REPO_COMMIT], check=True)
COMMIT = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert COMMIT == REPO_COMMIT, f'Expected pinned commit {REPO_COMMIT}, got {COMMIT}'
print('commit', COMMIT)
manifest = REPO / '.pageindex' / 'docbenchtr-pilot' / 'manifest.json'
assert manifest.exists(), 'Cached PageIndex trees are missing from the checkout.'
print('cached trees:', sorted(p.name for p in (REPO / '.pageindex' / 'docbenchtr-pilot' / 'docs').iterdir()))

In [ ]:
# 4. Exact Python environment from uv.lock (same package versions as the local run).
assert 'REPO' in globals(), 'Run the checkout cell first.'
!pip -q install uv
!cd /content/docbenchtr && uv sync --frozen --python 3.12 2>&1 | tail -n 5
PY = str(REPO / '.venv' / 'bin' / 'python')
!{PY} -c "import torch, pageindex; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"


In [ ]:
# 5. Ollama pinned to the local version, 32k context, same model + same Modelfile override.
import os, time, json, urllib.request
assert 'PY' in globals(), 'Run the environment cell first.'
OLLAMA_VERSION = '0.40.5'
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh > /tmp/ollama-install.log 2>&1; tail -n 3 /tmp/ollama-install.log
(DRIVE_ROOT / 'ollama_models').mkdir(parents=True, exist_ok=True)
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH='32768', OLLAMA_HOST='127.0.0.1:11434',
           OLLAMA_MODELS=str(DRIVE_ROOT / 'ollama_models'))
server = subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama-server.log', 'w'),
                          stderr=subprocess.STDOUT, env=env)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start; see /tmp/ollama-server.log')
os.environ.update({k: env[k] for k in ('OLLAMA_CONTEXT_LENGTH', 'OLLAMA_HOST', 'OLLAMA_MODELS')})
!ollama --version
!ollama pull qwen2.5:7b-instruct
# The local run served the model through this override (base blob sha256-2bada8a7...).
Path('/tmp/Modelfile').write_text('FROM qwen2.5:7b-instruct\nPARAMETER num_ctx 32768\nPARAMETER num_predict 4096\n')
!ollama create qwen2.5:7b-instruct -f /tmp/Modelfile
!ollama show qwen2.5:7b-instruct --modelfile | grep -E '^FROM|PARAMETER'


In [ ]:
# 6. Run the pilot. Output streams here and to Drive. Trees must be REUSED, never rebuilt.
assert 'server' in globals(), 'Run the Ollama cell first.'
log_path = DRIVE_ROOT / 'run_a100.log'
proc = subprocess.Popen([PY, '-u', '-m', 'docbenchtr.run_pilot'], cwd=REPO,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
reused = 0
with open(log_path, 'w') as log:
    for line in proc.stdout:
        log.write(line)
        log.flush()
        if 'Batches' in line:
            continue
        print(line, end='')
        if 'Reused verified PageIndex tree' in line:
            reused += 1
proc.wait()
print('exit code', proc.returncode)
assert proc.returncode == 0, f'Pilot failed; see {log_path}'
assert reused >= 3, 'Cached PageIndex trees were not reused: the run is not comparable to the local one.'

In [ ]:
# 7. Re-run any failed LLM Tree Search rows (same procedure as the local run).
rows = json.loads((REPO / 'results' / 'per_query.json').read_text())
failed = [r for r in rows if r['method'] == 'PageIndex LLM Tree Search' and r['status'] == 'failed']
print(len(failed), 'failed tree search rows')
if failed:
    subprocess.run([PY, '-u', '-m', 'docbenchtr.rerun_failed', 'PageIndex LLM Tree Search'], cwd=REPO, check=True)

In [ ]:
# 8. Record the environment, copy results to Drive, download a zip.
import shutil
from datetime import datetime, timezone
env_record = {
    'gpu': GPU_INFO, 'commit': COMMIT, 'ollama_version': OLLAMA_VERSION,
    'ollama_list': subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout,
    'modelfile': subprocess.run(['ollama', 'show', 'qwen2.5:7b-instruct', '--modelfile'], capture_output=True, text=True).stdout,
    'context_length': 32768, 'num_predict': 4096,
    'finished_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
}
(REPO / 'results' / 'run_environment.json').write_text(json.dumps(env_record, indent=2))
target = DRIVE_ROOT / 'results_a100_qwen2.5-7b'
if target.exists():
    shutil.rmtree(target)
shutil.copytree(REPO / 'results', target)
data_target = target / 'data'
data_target.mkdir(exist_ok=True)
for data_json in (REPO / 'data').glob('*.json'):
    shutil.copy2(data_json, data_target / data_json.name)
shutil.copy('/tmp/ollama-server.log', target / 'ollama-server.log')
archive = shutil.make_archive('/content/docbenchtr_results_a100', 'zip', target)
print('saved to', target)
from google.colab import files
files.download(archive)